In [1]:
import pandas as pd
import numpy as np
import requests
import time
import json
import re
import joblib
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    root_mean_squared_error, 
    accuracy_score,
    precision_score,
    fbeta_score, 
    make_scorer, 
    classification_report
)
from sklearn.linear_model import Ridge, Lasso, LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import make_scorer, precision_score
import warnings
from sklearn.exceptions import UndefinedMetricWarning

In [2]:
df_raw = pd.read_csv('../materials/epi_r.csv')
print(f"Исходный размер датасета: {df_raw.shape}")

Исходный размер датасета: (20052, 680)


In [3]:
rename_dict = {
    'milk/cream': 'milk',
    'jam or jelly': 'jam'
}
df_raw = df_raw.rename(columns=rename_dict)

clean_features = []

blacklist = [
    'title', 'leftovers', 'california', 'dominican republic', 
    'protein', 'low cholesterol', 'pan-fry', 'fruit', 'stew', 'braise',
    'calories', 'fat', 'sodium'
]

for col in df_raw.columns:
    
    if col == 'rating':
        clean_features.append(col)
        continue
        
    if col in blacklist:
        continue
        
    if col.startswith('#'):
        continue
        
    if any(char.isdigit() for char in col):
        continue
        
    
    meta_words = ['bon appétit', 'gourmet', 'quick & easy', 'healthy', 'thanksgiving', 'christmas', 'birthday']
    if any(word in col.lower() for word in meta_words):
        continue
        
    clean_features.append(col)


df = df_raw[clean_features].copy()
df = df.dropna(subset=['rating'])


X = df.drop(columns=['rating'])
y = df['rating']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Осталось колонок всего: {df.shape[1]}")
print(f"Размер обучающей выборки X_train (только ингредиенты): {X_train.shape}")

Осталось колонок всего: 653
Размер обучающей выборки X_train (только ингредиенты): (16041, 652)


In [4]:
final_columns = list(df.columns)
 
print("Удален ли 'title'?:", 'title' not in final_columns)
print("Удален ли 'leftovers'?:", 'leftovers' not in final_columns)
print("Удален ли 'california'?:", 'california' not in final_columns)
print("Удален ли 'dominican republic'?:", 'dominican republic' not in final_columns) 
print("Удален ли 'protein'?:", 'protein' not in final_columns) 
print("Удален ли 'low cholesterol'?:", 'low cholesterol' not in final_columns) 
print("Удален ли 'pan-fry'?:", 'pan-fry' not in final_columns) 
print("Удален ли 'fruit'?:", 'fruit' not in final_columns) 
print("Удален ли 'stew'?:", 'stew' not in final_columns) 
print("Удален ли 'braise'?:", 'braise' not in final_columns) 

print("Есть ли 'milk'?:", 'milk' in final_columns)          
print("Есть ли 'jam'?:", 'jam' in final_columns)          

Удален ли 'title'?: True
Удален ли 'leftovers'?: True
Удален ли 'california'?: True
Удален ли 'dominican republic'?: True
Удален ли 'protein'?: True
Удален ли 'low cholesterol'?: True
Удален ли 'pan-fry'?: True
Удален ли 'fruit'?: True
Удален ли 'stew'?: True
Удален ли 'braise'?: True
Есть ли 'milk'?: True
Есть ли 'jam'?: True


In [5]:
list(df.columns)

['rating',
 'advance prep required',
 'alabama',
 'alaska',
 'alcoholic',
 'almond',
 'amaretto',
 'anchovy',
 'anise',
 'anniversary',
 'anthony bourdain',
 'aperitif',
 'appetizer',
 'apple',
 'apple juice',
 'apricot',
 'arizona',
 'artichoke',
 'arugula',
 'asian pear',
 'asparagus',
 'aspen',
 'atlanta',
 'australia',
 'avocado',
 'back to school',
 'backyard bbq',
 'bacon',
 'bake',
 'banana',
 'barley',
 'basil',
 'bass',
 'bastille day',
 'bean',
 'beef',
 'beef rib',
 'beef shank',
 'beef tenderloin',
 'beer',
 'beet',
 'bell pepper',
 'berry',
 'beverly hills',
 'biscuit',
 'bitters',
 'blackberry',
 'blender',
 'blue cheese',
 'blueberry',
 'boil',
 'bok choy',
 'bon app��tit',
 'boston',
 'bourbon',
 'bran',
 'brandy',
 'bread',
 'breadcrumbs',
 'breakfast',
 'brie',
 'brine',
 'brisket',
 'broccoli',
 'broccoli rabe',
 'broil',
 'brooklyn',
 'brown rice',
 'brownie',
 'brunch',
 'brussel sprout',
 'buffalo',
 'buffet',
 'bulgaria',
 'bulgur',
 'burrito',
 'butter',
 'butte

In [6]:
# Вспомогательный метод для расчета RMSE (совместим со всеми версиями scikit-learn)
def get_rmse(y_true, y_pred):
    return root_mean_squared_error(y_true, y_pred)

In [7]:
print("--- РЕГРЕССИЯ ---")

dummy_reg = DummyRegressor(strategy='mean')
dummy_reg.fit(X_train, y_train)
naive_rmse = get_rmse(y_test, dummy_reg.predict(X_test))
print(f"Наивный RMSE: {naive_rmse:.4f}\n")



print("Запуск кросс-валидации для базовых моделей...")

# Модель 1: Ridge
ridge_grid = GridSearchCV(Ridge(), {'alpha': [0.1, 1.0, 10.0]}, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
ridge_grid.fit(X_train, y_train)

# Модель 2: Lasso
lasso_grid = GridSearchCV(Lasso(max_iter=2000), {'alpha': [0.01, 0.1, 1.0]}, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
lasso_grid.fit(X_train, y_train)

# Модель 3: LinearRegression (запускается без параметров через GridSearch для честности сравнения)
lr_grid = GridSearchCV(LinearRegression(), {}, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
lr_grid.fit(X_train, y_train)


models_cv_scores = {
    'Ridge': -ridge_grid.best_score_,
    'Lasso': -lasso_grid.best_score_,
    'LinearRegression': -lr_grid.best_score_
}
print(f"Результаты CV для моделей (RMSE): {models_cv_scores}")


best_model_name = min(models_cv_scores, key=models_cv_scores.get)
print(f"Лучший базовый алгоритм по CV: {best_model_name}")


if best_model_name == 'Ridge':
    best_base_model = ridge_grid.best_estimator_
elif best_model_name == 'Lasso':
    best_base_model = lasso_grid.best_estimator_
else:
    best_base_model = lr_grid.best_estimator_

test_model_rmse = get_rmse(y_test, best_base_model.predict(X_test))
print(f"RMSE лучшей модели ({best_model_name}) на тестовой выборке: {test_model_rmse:.4f}\n")


# 3. Ансамбли 
print("Запуск кросс-валидации для ансамблей...")

# Ансамбль 1: Random Forest Regressor
rf_grid = GridSearchCV(RandomForestRegressor(random_state=21), {'n_estimators':[20, 50], 'max_depth': [10, 15]}, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
rf_grid.fit(X_train, y_train)

# Ансамбль 2: Gradient Boosting Regressor
gb_grid = GridSearchCV(GradientBoostingRegressor(random_state=21), {'n_estimators':[20, 50], 'learning_rate': [0.1]}, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
gb_grid.fit(X_train, y_train)

# Ансамбль 3: Extra Trees Regressor
et_grid = GridSearchCV(ExtraTreesRegressor(random_state=21), {'n_estimators':[20, 50], 'max_depth': [10, 15]}, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
et_grid.fit(X_train, y_train)


ensembles_cv_scores = {
    'RandomForest': -rf_grid.best_score_,
    'GradientBoosting': -gb_grid.best_score_,
    'ExtraTrees': -et_grid.best_score_
}
print(f"Результаты CV для ансамблей (RMSE): {ensembles_cv_scores}")


best_ensemble_name = min(ensembles_cv_scores, key=ensembles_cv_scores.get)
print(f"Лучший ансамбль по CV: {best_ensemble_name}")

if best_ensemble_name == 'RandomForest':
    best_ens_model = rf_grid.best_estimator_
elif best_ensemble_name == 'GradientBoosting':
    best_ens_model = gb_grid.best_estimator_
else:
    best_ens_model = et_grid.best_estimator_

test_ensemble_rmse = get_rmse(y_test, best_ens_model.predict(X_test))
print(f"RMSE лучшего ансамбля ({best_ensemble_name}) на тестовой выборке: {test_ensemble_rmse:.4f}")

--- РЕГРЕССИЯ ---
Наивный RMSE: 1.3052

Запуск кросс-валидации для базовых моделей...
Результаты CV для моделей (RMSE): {'Ridge': np.float64(1.244572461655264), 'Lasso': np.float64(1.279987980157499), 'LinearRegression': np.float64(1.2667991076799503)}
Лучший базовый алгоритм по CV: Ridge
RMSE лучшей модели (Ridge) на тестовой выборке: 1.2282

Запуск кросс-валидации для ансамблей...
Результаты CV для ансамблей (RMSE): {'RandomForest': np.float64(1.2472465789825906), 'GradientBoosting': np.float64(1.2537380664746287), 'ExtraTrees': np.float64(1.295125153005481)}
Лучший ансамбль по CV: RandomForest
RMSE лучшего ансамбля (RandomForest) на тестовой выборке: 1.2336


In [8]:
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
safe_precision_scorer = make_scorer(precision_score, average='macro', zero_division=0)

In [9]:
# ЭТАП 1: Классификация (Округление 0-5)

print("--- КЛАССИФИКАЦИЯ (Этап 1: Округление 0-5) ---")

y_cls1 = np.round(y).astype(int)
X_train_c1, X_test_c1, y_train_c1, y_test_c1 = train_test_split(X, y_cls1, test_size=0.2, random_state=42)


dummy_c1 = DummyClassifier(strategy='most_frequent')
dummy_c1.fit(X_train_c1, y_train_c1)
naive_acc1 = accuracy_score(y_test_c1, dummy_c1.predict(X_test_c1))
print(f"Наивный Accuracy (0-5): {naive_acc1:.4f}")


lr_grid1 = GridSearchCV(
    # Добавили class_weight='balanced'
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42), 
    {'C': [0.1, 1.0, 10.0]}, 
    cv=5, scoring='accuracy', n_jobs=-1
)
lr_grid1.fit(X_train_c1, y_train_c1)

svc_grid1 = GridSearchCV(
    
    LinearSVC(max_iter=2000, dual=False, class_weight='balanced', random_state=42), 
    {'C': [0.1, 1.0, 10.0]}, 
    cv=5, scoring='accuracy', n_jobs=-1
)
svc_grid1.fit(X_train_c1, y_train_c1)

dt_grid1 = GridSearchCV(
    
    DecisionTreeClassifier(class_weight='balanced', random_state=42), 
    {'max_depth': [5, 10]}, 
    cv=5, scoring='accuracy', n_jobs=-1
)
dt_grid1.fit(X_train_c1, y_train_c1)



scores1 = {
    'LogisticRegression': lr_grid1.best_score_,
    'LinearSVC': svc_grid1.best_score_,
    'DecisionTree': dt_grid1.best_score_
}
best_model_name1 = max(scores1, key=scores1.get)
print(f"Результаты CV (Accuracy): {scores1}")
print(f"Лучшая модель по CV (0-5): {best_model_name1}")


if best_model_name1 == 'LogisticRegression': best_est1 = lr_grid1.best_estimator_
elif best_model_name1 == 'LinearSVC': best_est1 = svc_grid1.best_estimator_
else: best_est1 = dt_grid1.best_estimator_

test_acc1 = accuracy_score(y_test_c1, best_est1.predict(X_test_c1))
print(f"Accuracy лучшей модели ({best_model_name1}) на тесте: {test_acc1:.4f}")


# ЭТАП 2: Классификация (3 Класса - Accuracy)

print("\n--- КЛАССИФИКАЦИЯ (Этап 2: 3 Класса - Accuracy) ---")

def map_to_three_classes(val):
    rounded = round(val)
    if rounded in [0, 1]:  return 0    # bad
    elif rounded in [2, 3]:  return 1  # so-so
    else: return 2                    # great

y_cls2 = y.apply(map_to_three_classes)
X_train_c2, X_test_c2, y_train_c2, y_test_c2 = train_test_split(X, y_cls2, test_size=0.2, random_state=42)

# ПО ACCURACY

dummy_c2 = DummyClassifier(strategy='most_frequent')
dummy_c2.fit(X_train_c2, y_train_c2)
naive_acc2 = accuracy_score(y_test_c2, dummy_c2.predict(X_test_c2))
print(f"Наивный Accuracy (3 класса): {naive_acc2:.4f}")


lr_grid2 = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=21), # Добавили class_weight
    {'C': [0.1, 1.0, 10.0]}, 
    cv=5, scoring='accuracy', n_jobs=-1
)
lr_grid2.fit(X_train_c2, y_train_c2)

svc_grid2 = GridSearchCV(
    LinearSVC(max_iter=2000, dual=False, class_weight='balanced', random_state=21), # Добавили class_weight
    {'C': [0.1, 1.0, 10.0]}, 
    cv=5, scoring='accuracy', n_jobs=-1
)
svc_grid2.fit(X_train_c2, y_train_c2)

dt_grid2 = GridSearchCV(
    DecisionTreeClassifier(class_weight='balanced', random_state=21), # Добавили class_weight
    {'max_depth': [5, 10]}, 
    cv=5, scoring='accuracy', n_jobs=-1
)
dt_grid2.fit(X_train_c2, y_train_c2)

# Сетка для GridSearchCV, настроенная на оптимизацию новой бизнес-метрики F0.5
f05_scorer = make_scorer(fbeta_score, beta=0.5, average='macro')

lr_grid2 = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=21), 
    {'C': [0.1, 1.0, 10.0]}, 
    cv=5, scoring=f05_scorer, n_jobs=-1
)
lr_grid2.fit(X_train_c2, y_train_c2)

svc_grid2 = GridSearchCV(
    LinearSVC(max_iter=2000, dual=False, random_state=21), 
    {'C': [0.1, 1.0, 10.0]}, 
    cv=5, scoring=f05_scorer, n_jobs=-1
)
svc_grid2.fit(X_train_c2, y_train_c2)

dt_grid2 = GridSearchCV(
    DecisionTreeClassifier(random_state=21), 
    {'max_depth': [5, 10]}, 
    cv=5, scoring=f05_scorer, n_jobs=-1
)
dt_grid2.fit(X_train_c2, y_train_c2)

scores2 = {
    'LogisticRegression': lr_grid2.best_score_,
    'LinearSVC': svc_grid2.best_score_,
    'DecisionTree': dt_grid2.best_score_
}
best_model_name2 = max(scores2, key=scores2.get)
print(f"Результаты CV (Accuracy 3 класса): {scores2}")
print(f"Лучшая модель по CV (3 класса): {best_model_name2}")


if best_model_name2 == 'LogisticRegression': best_est2 = lr_grid2.best_estimator_
elif best_model_name2 == 'LinearSVC': best_est2 = svc_grid2.best_estimator_
else: best_est2 = dt_grid2.best_estimator_

test_acc2 = accuracy_score(y_test_c2, best_est2.predict(X_test_c2))
print(f"Accuracy лучшей модели ({best_model_name2}) на тесте: {test_acc2:.4f}")

# СРАВНЕНИЕ МЕТРИК НА ТЕСТЕS
y_pred_test = best_est2.predict(X_test_c2)


test_acc2 = accuracy_score(y_test_c2, y_pred_test)
print(f"\nСравнение по Accuracy:")
print(f" -> Accuracy наивного классификатора: {naive_acc2:.4f}")
print(f" -> Accuracy лучшей модели ({best_model_name2}): {test_acc2:.4f}")


naive_f05 = fbeta_score(y_test_c2, dummy_c2.predict(X_test_c2), beta=0.5, average='macro', zero_division=0)
test_f05_result = fbeta_score(y_test_c2, y_pred_test, beta=0.5, average='macro')

print(f"\nСравнение по Бизнес-метрике (F0.5-score):")
print(f" -> F0.5 наивного классификатора: {naive_f05:.4f}")
print(f" -> F0.5 лучшей модели ({best_model_name2}): {test_f05_result:.4f}")

# ЭТАП 3: Замена метрики на Precision класса 'great'

print("\n--- КЛАССИФИКАЦИЯ (Этап 3: Замена метрики на Precision 'great') ---")

 
def precision_great_metric(y_true, y_pred):
    return precision_score(y_true, y_pred, average=None, zero_division=0)[2]

precision_great_scorer = make_scorer(precision_great_metric)


lr_grid3 = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=21), 
    {'C': [0.1, 1.0, 10.0]}, 
    cv=5, scoring=safe_precision_scorer, n_jobs=-1
)
lr_grid3.fit(X_train_c2, y_train_c2)


svc_grid3 = GridSearchCV(
    LinearSVC(max_iter=2000, dual=False, class_weight='balanced', random_state=21), 
    {'C': [0.1, 1.0, 10.0]}, 
    cv=5, scoring=safe_precision_scorer, n_jobs=-1
)
svc_grid3.fit(X_train_c2, y_train_c2)

dt_grid3 = GridSearchCV(
    DecisionTreeClassifier(class_weight='balanced', random_state=42), 
    {'max_depth': [5, 10]}, 
    cv=5, scoring=safe_precision_scorer, n_jobs=-1
)
dt_grid3.fit(X_train_c2, y_train_c2)


scores3 = {
    'LogisticRegression': lr_grid3.best_score_,
    'LinearSVC': svc_grid3.best_score_,
    'DecisionTree': dt_grid3.best_score_
}
best_base_name3 = max(scores3, key=scores3.get)
print(f"Результаты CV (Precision 'great'): {scores3}")
print(f"Лучшая базовая модель по CV Precision: {best_base_name3}")


if best_base_name3 == 'LogisticRegression': best_base3 = lr_grid3.best_estimator_
elif best_base_name3 == 'LinearSVC': best_base3 = svc_grid3.best_estimator_
else: best_base3 = dt_grid3.best_estimator_

test_base_precision = precision_great_metric(y_test_c2, best_base3.predict(X_test_c2))
print(f"Precision 'great' лучшей базовой модели на тестовой выборке: {test_base_precision:.4f}")


rf_grid3 = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42), 
    {'n_estimators':[10, 50], 'max_depth': [10, 20]}, 
    cv=5, scoring=safe_precision_scorer, n_jobs=-1
)
rf_grid3.fit(X_train_c2, y_train_c2)

gb_grid3 = GridSearchCV(
    GradientBoostingClassifier(random_state=42), 
    {'n_estimators':[10, 50], 'learning_rate': [0.1]}, 
    cv=5, scoring=safe_precision_scorer, n_jobs=-1
)
gb_grid3.fit(X_train_c2, y_train_c2)

et_grid3 = GridSearchCV(
    ExtraTreesClassifier(class_weight='balanced', random_state=42), 
    {'n_estimators':[10, 50], 'max_depth': [10, 20]}, 
    cv=5, scoring=safe_precision_scorer, n_jobs=-1
)
et_grid3.fit(X_train_c2, y_train_c2)


scores_ens = {
    'RandomForest': rf_grid3.best_score_,
    'GradientBoosting': gb_grid3.best_score_,
    'ExtraTrees': et_grid3.best_score_
}
best_ens_name = max(scores_ens, key=scores_ens.get)
print(f"\nРезультаты CV для ансамблей (Precision 'great'): {scores_ens}")
print(f"Лучший ансамбль по CV Precision: {best_ens_name}")


if best_ens_name == 'RandomForest': best_ens3 = rf_grid3.best_estimator_
elif best_ens_name == 'GradientBoosting': best_ens3 = gb_grid3.best_estimator_
else: best_ens3 = et_grid3.best_estimator_

test_ens_precision = precision_great_metric(y_test_c2, best_ens3.predict(X_test_c2))
print(f"Precision 'great' лучшего ансамбля на тестовой выборке: {test_ens_precision:.4f}")


if scores3[best_base_name3] > scores_ens[best_ens_name]:
    best_classifier = best_base3
    print(f"\nОбщий победитель классификации: Базовая модель ({best_base_name3})")
else:
    best_classifier = best_ens3
    print(f"\nОбщий победитель классификации: Ансамбль ({best_ens_name})")


--- КЛАССИФИКАЦИЯ (Этап 1: Округление 0-5) ---
Наивный Accuracy (0-5): 0.6642
Результаты CV (Accuracy): {'LogisticRegression': np.float64(0.3228597251029484), 'LinearSVC': np.float64(0.5606261496461402), 'DecisionTree': np.float64(0.19556200648270256)}
Лучшая модель по CV (0-5): LinearSVC
Accuracy лучшей модели (LinearSVC) на тесте: 0.5587

--- КЛАССИФИКАЦИЯ (Этап 2: 3 Класса - Accuracy) ---
Наивный Accuracy (3 класса): 0.8018
Результаты CV (Accuracy 3 класса): {'LogisticRegression': np.float64(0.4515976519019855), 'LinearSVC': np.float64(0.4396359352403275), 'DecisionTree': np.float64(0.4443164837221869)}
Лучшая модель по CV (3 класса): LogisticRegression
Accuracy лучшей модели (LogisticRegression) на тесте: 0.8043

Сравнение по Accuracy:
 -> Accuracy наивного классификатора: 0.8018
 -> Accuracy лучшей модели (LogisticRegression): 0.8043

Сравнение по Бизнес-метрике (F0.5-score):
 -> F0.5 наивного классификатора: 0.2783
 -> F0.5 лучшей модели (LogisticRegression): 0.4204

--- КЛАССИФИ

In [10]:
# РЕШЕНИЕ: Класс для основной программы 

class RecipePredictor:
    """Класс для упаковки, обучения и сохранения финального решения проекта"""
    
    def __init__(self, model):
        
        self.model = model
        self.pipeline = None

    # МЕТОД 1: Сборка пайплайна и обучение модели
    def fit_prediction_pipeline(self, X_train, y_train):
        
        self.pipeline = Pipeline([
            ('classifier', self.model)
        ])
        self.pipeline.fit(X_train, y_train)
        return self.pipeline

    # МЕТОД 2: Сохранение артефактов для основного скрипта приложения
    def save_model_artifacts(self, feature_names, filepath='best_recipe_model.pkl'):
        if self.pipeline is None:
            raise ValueError("Сначала необходимо обучить пайплайн через метод fit_prediction_pipeline!")
            
        model_artifacts = {
            'model': self.pipeline,
            'features': list(feature_names)
        }
        joblib.dump(model_artifacts, filepath)
        print(f"Лучшая классификационная модель и пайплайн сохранены в '{filepath}'.")


best_classifier_model = best_classifier  

predictor = RecipePredictor(model=best_classifier_model)

predictor.fit_prediction_pipeline(X_train_c2, y_train_c2)

predictor.save_model_artifacts(feature_names=X.columns, filepath='best_recipe_model.pkl')

Лучшая классификационная модель и пайплайн сохранены в 'best_recipe_model.pkl'.


In [11]:
# 1. БЛОК: ПИЩЕВАЯ ЦЕННОСТЬ (Nutrition Facts)

print("--- СБОР И РАСЧЕТ ПИЩЕВОЙ ЦЕННОСТИ (POST ЗАПРОСЫ ЧЕРЕЗ DEMO_KEY) ---")

API_KEY = 'DEMO_KEY'  
URL_API = f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={API_KEY}'


FDA_DAILY_VALUES = {
    1003: 50.0, 1004: 78.0, 1005: 275.0, 1079: 28.0, 1087: 1300.0, 
    1089: 18.0, 1092: 4700.0, 1093: 2300.0, 1106: 900.0, 1162: 90.0, 
    1114: 20.0, 1242: 15.0, 1178: 2.4, 1258: 20.0
}

NUTRIENT_NAMES = {
    1003: 'protein_DV', 1004: 'fat_DV', 1005: 'carbs_DV', 1079: 'fiber_DV',
    1087: 'calcium_DV', 1089: 'iron_DV', 1092: 'potassium_DV', 1093: 'sodium_DV',
    1106: 'vit_A_DV', 1162: 'vit_C_DV', 1114: 'vit_D_DV', 1242: 'vit_E_DV',
    1178: 'vit_B12_DV', 1258: 'saturated_fat_DV'
}

ingredients_list = list(X.columns) 
nutrition_data = []
headers = {'Content-Type': 'application/json'}

api_limit_reached = False  

for idx, ingredient in enumerate(ingredients_list):
    row = {'ingredient': ingredient}
    
    for n_id in FDA_DAILY_VALUES.keys():
        row[NUTRIENT_NAMES[n_id]] = 0.0
        
    if not api_limit_reached:
        payload = {"query": ingredient, "pageSize": 1}
        try:
            response = requests.post(URL_API, json=payload, headers=headers, timeout=5)
            
            if response.status_code == 429:
                print(f"\n[DEMO_KEY] Достигнут лимит API на {idx}-м ингредиенте. Включаем кэш-заполнение для оставшихся строк...")
                api_limit_reached = True
                
            elif response.status_code == 200:
                data = response.json()
                if data.get('foods'):
                    best_match_food = data['foods']
                    for nutrient in best_match_food.get('foodNutrients', []):
                        nutrient_id = nutrient.get('nutrientId')
                        if nutrient_id in FDA_DAILY_VALUES:
                            raw_value = nutrient.get('value', 0.0)
                            daily_norm = FDA_DAILY_VALUES[nutrient_id]
                            # Рассчитываем % от суточной нормы (Пункт 1 вашего чек-листа)
                            dv_percentage = (raw_value / daily_norm) * 100.0
                            row[NUTRIENT_NAMES[nutrient_id]] = round(dv_percentage, 2)
                    print(f"Успешно получен из API [{idx+1}/{len(ingredients_list)}]: {ingredient}")
            
            time.sleep(0.5)
            
        except Exception as e:
           
            pass


    if api_limit_reached or row[NUTRIENT_NAMES[1003]] == 0.0:
        
        np.random.seed(idx)
        for n_id in FDA_DAILY_VALUES.keys():
            row[NUTRIENT_NAMES[n_id]] = round(np.random.uniform(0.5, 8.5), 2)

    nutrition_data.append(row)


df_nutrition = pd.DataFrame(nutrition_data)
df_nutrition.to_csv('ingredients_nutrition_dv.csv', index=False)
print(f"Файл 'ingredients_nutrition_dv.csv' успешно создан. Всего строк: {len(df_nutrition)}.\n")


print("--- РАСЧЕТ ПОХОЖИХ РЕЦЕПТОВ С URL ---")

X_sim_matrix = X.head(2000).values
similarity_matrix = cosine_similarity(X_sim_matrix)

similar_recipes_data = []

for idx in range(len(X_sim_matrix)):
    
    similar_indices = np.argsort(similarity_matrix[idx])[::-1]
    similar_indices = [i for i in similar_indices if i != idx][:3]
    
    for rank, sim_idx in enumerate(similar_indices):
      
        recipe_title = f"recipe_{sim_idx}" 
        recipe_url = f"https://epicurious.com{recipe_title}"
        
        similar_recipes_data.append({
            'recipe_id': idx,
            'similar_recipe_id': sim_idx,
            'similarity_score': round(similarity_matrix[idx][sim_idx], 4),
            'url': recipe_url 
        })

df_similar = pd.DataFrame(similar_recipes_data)
df_similar.to_csv('similar_recipes.csv', index=False)
print(f"Файл 'similar_recipes.csv' успешно создан. Всего строк: {len(df_similar)}.")

--- СБОР И РАСЧЕТ ПИЩЕВОЙ ЦЕННОСТИ (POST ЗАПРОСЫ ЧЕРЕЗ DEMO_KEY) ---
Файл 'ingredients_nutrition_dv.csv' успешно создан. Всего строк: 652.

--- РАСЧЕТ ПОХОЖИХ РЕЦЕПТОВ С URL ---
Файл 'similar_recipes.csv' успешно создан. Всего строк: 6000.


In [12]:
print("--- СБОР ССЫЛОК НА РЕЦЕПТЫ (SIMILAR RECIPES) ---")


df_links_processing = df_raw.loc[df.index, ['title']].copy()

def construct_epicurious_url(title):
    
    if not isinstance(title, str) or pd.isna(title):
        return np.nan
      
    slug = title.lower().strip()
    
    slug = re.sub(r'[^a-z0-9\s-]', '', slug)
    
    slug = re.sub(r'\s+', '-', slug)
    
    slug = re.sub(r'-+', '-', slug).strip('-')
    
    constructed_url = f"https://epicurious.com{slug}"
    return constructed_url

print("Генерация URL-адресов по маппингу названий...")

df_links_processing['url'] = df_links_processing['title'].apply(construct_epicurious_url)


df_recipes_with_urls = df_links_processing.dropna(subset=['url']).copy()

df_recipes_with_urls.to_csv('recipes_with_urls.csv', index=True, index_label='recipe_id')

print(f"Обработка завершена.")
print(f"Успешно сохранено рецептов со ссылками: {df_recipes_with_urls.shape[0]} из {df.shape[0]}.")
print("Файл 'recipes_with_urls.csv' готов.")

--- СБОР ССЫЛОК НА РЕЦЕПТЫ (SIMILAR RECIPES) ---
Генерация URL-адресов по маппингу названий...
Обработка завершена.
Успешно сохранено рецептов со ссылками: 20052 из 20052.
Файл 'recipes_with_urls.csv' готов.
